In [1]:
import yfinance as yf

df = yf.download("^NSEI", start="2010-01-01", progress=False)
print(df.shape)
print(df.head())
print(df.tail())
print(df.isna().sum())

(4128, 5)
Price             Close         High          Low         Open Volume
Ticker            ^NSEI        ^NSEI        ^NSEI        ^NSEI  ^NSEI
Date                                                                 
2010-01-04  5232.200195  5238.450195  5167.100098  5200.899902      0
2010-01-05  5277.899902  5288.350098  5242.399902  5277.149902      0
2010-01-06  5281.799805  5310.850098  5260.049805  5278.149902      0
2010-01-07  5263.100098  5302.549805  5244.750000  5281.799805      0
2010-01-08  5244.750000  5276.750000  5234.700195  5264.250000      0
Price              Close          High           Low          Open  Volume
Ticker             ^NSEI         ^NSEI         ^NSEI         ^NSEI   ^NSEI
Date                                                                      
2026-10-16  23146.150391  23149.849609  22913.800781  23065.099609  391600
2026-10-19  22888.949219  23175.250000  22845.000000  23150.849609  285800
2026-10-21  22811.349609  23120.900391  22649.900391  2

e:\Github\Projects\sip-vs-lumpsum\.venv\Lib\site-packages\yfinance\utils.py:568: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  quotes.sort_index(inplace=True)
e:\Github\Projects\sip-vs-lumpsum\.venv\Lib\site-packages\yfinance\utils.py:514: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.drop(
e:\Github\Projects\sip-vs-lumpsum\.venv\Lib\site-packages\yfinance\utils.py:518: Pandas4Warning: The inplace keyword in DataFrame.rename is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.rename(columns={
e:\Github\Projects\sip-vs-lumpsum\

In [2]:
import warnings
warnings.filterwarnings("ignore")

In [3]:
df.columns = df.columns.get_level_values(0)
print(df.columns.tolist())

['Close', 'High', 'Low', 'Open', 'Volume']


In [4]:
close = df["Close"]
monthly = close.groupby(close.index.to_period("M")).agg(["first", "mean", "last"])
print(monthly.shape)
print(monthly.head())
print(monthly.tail())

(202, 3)
               first         mean         last
Date                                          
2010-01  5232.200195  5156.223684  4882.049805
2010-02  4899.700195  4843.905273  4922.299805
2010-03  5017.000000  5178.147624  5249.100098
2010-04  5290.500000  5294.755054  5278.000000
2010-05  5222.750000  5052.971401  5086.299805
                first          mean          last
Date                                             
2026-06  23382.599609  23683.145182  23865.750000
2026-07  24005.849609  24133.632473  24383.599609
2026-08  24774.300781  24361.273996  24080.400391
2026-09  24055.800781  23377.468936  22620.449219
2026-10  22421.949219  22663.236458  22600.000000


In [5]:
from pathlib import Path

Path("../data/raw").mkdir(parents=True, exist_ok=True)
df[["Close"]].to_csv("../data/raw/nifty50_close.csv")

In [6]:
import pandas as pd

check = pd.read_csv("../data/raw/nifty50_close.csv", index_col="Date", parse_dates=True)
print(check.shape)
print(check.index.dtype)
print(check.tail(2))

(4128, 1)
datetime64[us]
                   Close
Date                    
2026-10-22  22687.650391
2026-10-23  22600.000000


In [7]:
s = check["Close"]

print("sorted:", s.index.is_monotonic_increasing)
print("duplicate dates:", s.index.duplicated().sum())
print("prices <= 0:", (s <= 0).sum())

gaps = s.index.to_series().diff().dt.days
print("longest gap (days):", gaps.max(), "ending", gaps.idxmax().date())

ret = s.pct_change().dropna()
print(ret.nsmallest(3))
print(ret.nlargest(3))

sorted: True
duplicate dates: 0
prices <= 0: 0
longest gap (days): 6.0 ending 2014-10-07
Date
2020-03-23   -0.129805
2020-03-12   -0.083019
2020-03-16   -0.076121
Name: Close, dtype: float64
Date
2020-04-07    0.087632
2020-03-25    0.066247
2020-03-20    0.058329
Name: Close, dtype: float64


In [16]:
def monthly_buy_prices(close, sip_day=1):
    rows = []
    for period in close.index.to_period("M").unique():
        month = close[close.index.to_period("M") == period]
        eligible = month[month.index.day >= sip_day]
        if len(eligible) == 0:
            continue
        rows.append({"month": period,
                     "buy_date": eligible.index[0],
                     "price": eligible.iloc[0]})
    return pd.DataFrame(rows).set_index("month")

In [17]:
b13 = monthly_buy_prices(close, sip_day=13)
print(b13.head(6))
print(b13[b13["buy_date"].dt.day != 13].head())

          buy_date        price
month                          
2010-01 2010-01-13  5233.950195
2010-02 2010-02-15  4801.950195
2010-03 2010-03-15  5128.899902
2010-04 2010-04-13  5322.950195
2010-05 2010-05-13  5178.899902
2010-06 2010-06-14  5197.700195
          buy_date        price
month                          
2010-02 2010-02-15  4801.950195
2010-03 2010-03-15  5128.899902
2010-06 2010-06-14  5197.700195
2010-11 2010-11-15  6121.600098
2011-02 2011-02-14  5456.000000


In [18]:
Path("../data/processed").mkdir(parents=True, exist_ok=True)
monthly_buy_prices(close, sip_day=1).to_csv("../data/processed/buy_prices_day1.csv")